# Setup

In [6]:
import sys, json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

sys.path.append('.')  # repo root, where common.py lives
import common as c

ROOT = Path('.')
RESULTS = ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# Load the one and only model used all tournament -- no retraining happened
# mid-tournament, so every prediction we check came from this exact file.
model, scaler, T = c.load_model()
print('Loaded football_v2.pth')
print('Temperature:', round(T, 3))

# Pull every match, then split into "finished" (real results exist),
# then further into group-stage vs knockout -- graded separately since
# they're different questions (see Test 1 vs Test 2).
games = c.fetch_wc26('games')
teams = c.fetch_wc26('teams')
team_lookup = c.build_team_lookup(teams)

finished = [g for g in games if str(g.get('finished', '')).upper() == 'TRUE']
group_games = [g for g in finished if (g.get('type') or '').lower() == 'group']
# Fallback if common.py does not define ROUND_ORDER
knockout_rounds = getattr(c, 'ROUND_ORDER', [
    'round of 16',
    'quarter final',
    'semi final',
    'third place',
    'final',
])

normalized_rounds = {str(r).lower().replace('_', ' ') for r in knockout_rounds}
knockout_games = [
    g for g in finished
    if str(g.get('type') or '').lower().replace('_', ' ') in normalized_rounds
]

print(f'Total finished matches: {len(finished)}')
print(f'  Group stage: {len(group_games)}')
print(f'  Knockout   : {len(knockout_games)}')

2026-07-23 10:15:17.143 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loaded football_v2.pth
Temperature: 0.839


2026-07-23 10:15:18.497 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Total finished matches: 104
  Group stage: 72
  Knockout   : 1


# First test



In [7]:
# For every group-stage match, pull the PRE-GAME prediction -- the very
# first point on that match's History-page chart, at minute 0, before a
# single goal has happened. build_match_timeline already builds this
# whole curve for us; we just need its first entry.
def pregame_prediction(game):
    timeline = c.build_match_timeline(game, team_lookup, model, scaler, T)
    return timeline[0]  # minute 0 is always the first checkpoint

rows = []
skipped = 0
for g in group_games:
    home_id, away_id = g.get('home_team_id'), g.get('away_team_id')
    if home_id not in team_lookup or away_id not in team_lookup:
        skipped += 1
        continue
    pre = pregame_prediction(g)
    rows.append({
        'home': team_lookup[home_id]['fifa_code'],
        'away': team_lookup[away_id]['fifa_code'],
        'p_home': pre['p_home'], 'p_draw': pre['p_draw'], 'p_away': pre['p_away'],
        'final_home_score': c.safe_int(g.get('home_score')),
        'final_away_score': c.safe_int(g.get('away_score')),
    })

group_df = pd.DataFrame(rows)
print(f'Reconstructed pre-game predictions for {len(group_df)} group matches (skipped {skipped}).')

# What actually happened, in plain terms: home won, away won, or a draw.
def actual_outcome(row):
    if row.final_home_score > row.final_away_score: return 'home'
    if row.final_home_score < row.final_away_score: return 'away'
    return 'draw'

group_df['actual'] = group_df.apply(actual_outcome, axis=1)

# The model's pick is just whichever of the three numbers was biggest.
group_df['predicted'] = group_df[['p_home', 'p_draw', 'p_away']].idxmax(axis=1).str.replace('p_', '')
group_df['correct'] = group_df['predicted'] == group_df['actual']

group_acc = group_df['correct'].mean()
baseline_acc = (group_df['actual'] == 'home').mean()  # "always guess home" baseline

print(f'\nGroup stage accuracy     : {group_acc:.3f}')
print(f'Baseline (always "home") : {baseline_acc:.3f}')
print(f'Phase 3 kickoff baseline : 0.574  (for comparison against a past validation)')

print('\nOutcome breakdown (rows = what actually happened, columns = what we predicted):')
print(pd.crosstab(group_df['actual'], group_df['predicted']))

Reconstructed pre-game predictions for 72 group matches (skipped 0).

Group stage accuracy     : 0.556
Baseline (always "home") : 0.472
Phase 3 kickoff baseline : 0.574  (for comparison against a past validation)

Outcome breakdown (rows = what actually happened, columns = what we predicted):
predicted  away  draw  home
actual                     
away         16     0     2
draw          8     0    12
home          9     1    24


# Test 2: Knockout stage accuracy (2-way: who advances)

In [8]:
# Broader net than a fixed list of stage names -- "not group" catches
# everything, including Round of 32, which our original stage-name list
# had missed. This also prints exactly what stage names actually exist,
# so we're not guessing blind a second time.
knockout_games_all = [g for g in finished if (g.get('type') or '').lower() != 'group']
print('Stage names actually found:', sorted(set((g.get('type') or '?').lower() for g in knockout_games_all)))
print(f'Total knockout matches: {len(knockout_games_all)}')

rows = []
skipped = 0
for g in knockout_games_all:
    home_id, away_id = g.get('home_team_id'), g.get('away_team_id')
    if home_id not in team_lookup or away_id not in team_lookup:
        skipped += 1
        continue
    pre = pregame_prediction(g)  # reused from Test 1's cell
    # Fallback: determine the advancing team directly in the notebook
    # when common.determine_ko_winner_id is unavailable.
    home_pen = c.safe_int(g.get('home_penalty_score'))
    away_pen = c.safe_int(g.get('away_penalty_score'))
    home_score = c.safe_int(g.get('home_score'))
    away_score = c.safe_int(g.get('away_score'))

    # Penalties take precedence if both sides have a recorded penalty score.
    if home_pen is not None and away_pen is not None:
        winner_id = home_id if home_pen > away_pen else away_id
    else:
        # Otherwise, use the score result.
        winner_id = home_id if home_score > away_score else away_id
    if winner_id not in team_lookup:
        skipped += 1
        continue
    rows.append({
        'type': (g.get('type') or '?').lower(),
        'home': team_lookup[home_id]['fifa_code'],
        'away': team_lookup[away_id]['fifa_code'],
        'p_home': pre['p_home'], 'p_draw': pre['p_draw'],
        'actual_winner': team_lookup[winner_id]['fifa_code'],
    })

ko_df = pd.DataFrame(rows)
print(f'\nGraded {len(ko_df)} knockout matches (skipped {skipped}).')

# Same draw-split rule the live Bracket page used: a knockout match can't
# stay level, so we split the draw chance 50/50 between the two teams.
ko_df['p_home_advance'] = ko_df['p_home'] + ko_df['p_draw'] / 2
ko_df['predicted_winner'] = np.where(ko_df['p_home_advance'] > 0.5, ko_df['home'], ko_df['away'])
ko_df['correct'] = ko_df['predicted_winner'] == ko_df['actual_winner']

ko_acc = ko_df['correct'].mean()
print(f'\nOverall knockout accuracy: {ko_acc:.3f}   (baseline coin-flip: 0.500)')

print('\nBy stage:')
for stage, sub in ko_df.groupby('type'):
    print(f'  {stage:<10} {len(sub):>3} matches   accuracy {sub["correct"].mean():.3f}')

Stage names actually found: ['final', 'qf', 'r16', 'r32', 'sf', 'third']
Total knockout matches: 32

Graded 32 knockout matches (skipped 0).

Overall knockout accuracy: 0.031   (baseline coin-flip: 0.500)

By stage:
  final        1 matches   accuracy 0.000
  qf           4 matches   accuracy 0.000
  r16          8 matches   accuracy 0.125
  r32         16 matches   accuracy 0.000
  sf           2 matches   accuracy 0.000
  third        1 matches   accuracy 0.000
